# Audio Pipeline: From Mouth to Whisper Voice Conversion

> **Goal:** Understand how audio is recorded, processed, and converted from normal speech to whisper using a Mel-to-Mel neural network.

---

## What we will cover:

**Part I — Audio Basics**
- How audio recording works (data type, data representation, data flow)
- Audio normalization — before & after plots
- What is a Mel spectrogram?

**Part II — Full Pipeline**
1. Dataset preparation
2. Feature extraction
3. Mel-Mel model
4. Training
5. Inference
6. Vocoder

In [1]:
# install what we need if not already installed
# !pip install numpy matplotlib scipy librosa torch

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.io import wavfile
from scipy.signal import spectrogram
import warnings
warnings.filterwarnings('ignore')

# set a nice style
plt.rcParams['figure.figsize'] = (14, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('all imports OK!')

all imports OK!


---
# PART I — Audio Basics

## 1. How Audio Recording Works

### The Data Flow: Mouth → Microphone → Computer

```
  Your mouth
      |  (sound waves / air pressure changes)
      v
  Microphone
      |  (converts pressure → analog voltage signal)
      v
  ADC (Analog-to-Digital Converter — built into your sound card)
      |  samples the voltage at a fixed rate (e.g. 16,000 times per second)
      v
  Digital samples  →  stored as int16 or float32 numbers in a .wav file
```

### Key concepts

| Concept | What it means | Typical value |
|---|---|---|
| **Sample Rate** | How many snapshots per second | 16,000 Hz (speech), 44,100 Hz (music) |
| **Bit Depth** | How precise each snapshot is | 16-bit → values from -32768 to +32767 |
| **Channels** | Mono vs Stereo | 1 (mono) for speech |
| **Duration** | seconds = total_samples / sample_rate | e.g. 3 sec = 48,000 samples at 16kHz |

### Data types in Python

- **int16** — raw WAV format, values in [-32768, 32767]
- **float32** — normalized version, values in [-1.0, 1.0] (what we use in ML)

In [ ]:
# ============================================================
# PRACTICAL EXAMPLE: Simulate a recorded audio signal
# (we generate a fake speech-like signal since we have no mic)
# ============================================================

SAMPLE_RATE = 16000   # 16 kHz -- standard for speech
DURATION    = 2.0     # 2 seconds

t = np.linspace(0, DURATION, int(SAMPLE_RATE * DURATION))

# simulate speech: mix of a few sine waves at speech frequencies
# (real speech is much more complex, but this gives the idea)
signal_float = (
    0.5  * np.sin(2 * np.pi * 200  * t) +   # 200 Hz  -- low vowel energy
    0.3  * np.sin(2 * np.pi * 800  * t) +   # 800 Hz  -- first formant
    0.2  * np.sin(2 * np.pi * 2000 * t) +   # 2000 Hz -- second formant
    0.05 * np.random.randn(len(t))           # noise (like microphone noise)
)

# add a silent tail (simulating "forgot to stop recording" problem)
silence = np.zeros(SAMPLE_RATE)   # 1 second of silence at the end
signal_with_silence = np.concatenate([signal_float, silence])

# convert to int16 (as stored in a .wav file)
signal_int16 = (signal_float * 32767).astype(np.int16)

print(f'Sample rate : {SAMPLE_RATE} Hz')
print(f'Duration    : {DURATION} seconds')
print(f'Total samples: {len(signal_float)}')
print(f'Data type (raw WAV)   : {signal_int16.dtype}  | range: [{signal_int16.min()}, {signal_int16.max()}]')
print(f'Data type (for ML)    : {signal_float.dtype} | range: [{signal_float.min():.3f}, {signal_float.max():.3f}]')
print()
print('First 10 samples (int16):', signal_int16[:10])
print('First 10 samples (float):', np.round(signal_float[:10], 4))

In [ ]:
# ---- plot the waveform ----
fig, axes = plt.subplots(1, 2, figsize=(15, 3))

# left: show first 0.02 sec (20 ms) -- you can see individual samples
n_show = int(0.02 * SAMPLE_RATE)
axes[0].plot(t[:n_show] * 1000, signal_int16[:n_show], color='steelblue', linewidth=1)
axes[0].set_xlabel('Time (ms)')
axes[0].set_ylabel('Amplitude (int16)')
axes[0].set_title('Zoomed in: first 20 ms  (you can see individual samples)')

# right: full 2-second waveform
axes[1].plot(t, signal_float, color='steelblue', linewidth=0.5)
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Amplitude (float32)')
axes[1].set_title('Full waveform — 2 seconds of simulated speech')

plt.tight_layout()
plt.show()

print('notice: the waveform is just a list of numbers -- one per sample!')

---
## 2. Audio Normalization

**Why normalize?**

Different recordings have different volumes. If we feed raw audio to a neural network, loud files dominate, quiet files are ignored. Normalization fixes this.

**Two common methods:**

| Method | Formula | What it does |
|---|---|---|
| **Peak normalization** | `x / max(abs(x))` | Sets loudest sample to ±1.0 |
| **RMS normalization** | `x * (target_rms / rms(x))` | Sets average loudness to a target level |

In [ ]:
# ============================================================
# PRACTICAL EXAMPLE: Normalization before vs after
# ============================================================

# --- create a quiet signal (as if someone whispered far from the mic) ---
quiet_signal = signal_float * 0.08   # very quiet: amplitude is 8% of original

# --- BEFORE normalization ---
rms_before = np.sqrt(np.mean(quiet_signal ** 2))
peak_before = np.max(np.abs(quiet_signal))
print(f'BEFORE normalization:')
print(f'  Peak amplitude : {peak_before:.4f}')
print(f'  RMS amplitude  : {rms_before:.4f}')

# --- method 1: peak normalization ---
peak_normalized = quiet_signal / np.max(np.abs(quiet_signal))

# --- method 2: RMS normalization ---
target_rms = 0.1   # we want RMS = 0.1
rms_normalized = quiet_signal * (target_rms / rms_before)

print(f'\nAFTER peak normalization:')
print(f'  Peak amplitude : {np.max(np.abs(peak_normalized)):.4f}  (always = 1.0)')
print(f'  RMS amplitude  : {np.sqrt(np.mean(peak_normalized**2)):.4f}')

print(f'\nAFTER RMS normalization (target = {target_rms}):')
print(f'  Peak amplitude : {np.max(np.abs(rms_normalized)):.4f}')
print(f'  RMS amplitude  : {np.sqrt(np.mean(rms_normalized**2)):.4f}  (= target)')

In [ ]:
# ---- plot: before vs after normalization ----
t_plot = np.linspace(0, DURATION, len(quiet_signal))

fig, axes = plt.subplots(1, 3, figsize=(16, 3), sharey=False)

axes[0].plot(t_plot, quiet_signal, color='gray', linewidth=0.5)
axes[0].set_title('BEFORE normalization (very quiet signal)')
axes[0].set_ylabel('Amplitude')
axes[0].set_ylim(-1.1, 1.1)
axes[0].axhline(0, color='black', linewidth=0.5)

axes[1].plot(t_plot, peak_normalized, color='steelblue', linewidth=0.5)
axes[1].set_title('AFTER peak normalization')
axes[1].set_ylim(-1.1, 1.1)
axes[1].axhline(0, color='black', linewidth=0.5)

axes[2].plot(t_plot, rms_normalized, color='green', linewidth=0.5)
axes[2].set_title(f'AFTER RMS normalization (target RMS={target_rms})')
axes[2].set_ylim(-1.1, 1.1)
axes[2].axhline(0, color='black', linewidth=0.5)

for ax in axes:
    ax.set_xlabel('Time (s)')

plt.suptitle('Audio Normalization — Before vs After', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Mel Spectrogram — The "image" of audio

A raw waveform is hard for a neural network to learn from.
Instead we convert audio into a **Mel spectrogram** — a 2D image where:

- **X-axis** = time (frames)
- **Y-axis** = frequency (Mel scale — matches how humans perceive pitch)
- **Color** = energy (how loud each frequency is at that moment)

**Steps to compute it:**

```
waveform
   | STFT (Short-Time Fourier Transform — slide a window, compute FFT each time)
   v
Power spectrogram  [freq x time]
   | Mel filter bank (group frequencies into ~80 mel bands)
   v
Mel spectrogram  [80 x time]
   | log (so values don't span 6 orders of magnitude)
   v
Log-Mel spectrogram  <- this is what we feed to the model
```

In [ ]:
# ============================================================
# PRACTICAL EXAMPLE: Compute Mel spectrogram from scratch
# (using only numpy/scipy so no extra installs needed)
# ============================================================

def hz_to_mel(hz):
    """convert frequency in Hz to Mel scale"""
    return 2595 * np.log10(1 + hz / 700.0)

def mel_to_hz(mel):
    """convert Mel scale back to Hz"""
    return 700 * (10 ** (mel / 2595.0) - 1)

def mel_filterbank(n_filters, n_fft, sample_rate, fmin=0, fmax=None):
    """create a mel filterbank matrix: shape (n_filters, n_fft//2+1)"""
    if fmax is None:
        fmax = sample_rate / 2
    
    # evenly spaced points on mel scale
    mel_min = hz_to_mel(fmin)
    mel_max = hz_to_mel(fmax)
    mel_points = np.linspace(mel_min, mel_max, n_filters + 2)
    hz_points  = mel_to_hz(mel_points)
    
    # convert to FFT bin indices
    bins = np.floor((n_fft + 1) * hz_points / sample_rate).astype(int)
    
    fb = np.zeros((n_filters, n_fft // 2 + 1))
    for m in range(1, n_filters + 1):
        f_m_minus = bins[m - 1]
        f_m       = bins[m]
        f_m_plus  = bins[m + 1]
        for k in range(f_m_minus, f_m):
            fb[m-1, k] = (k - bins[m-1]) / (bins[m] - bins[m-1] + 1e-8)
        for k in range(f_m, f_m_plus):
            fb[m-1, k] = (bins[m+1] - k) / (bins[m+1] - bins[m] + 1e-8)
    return fb

def compute_mel_spectrogram(signal, sr, n_mels=80, n_fft=512, hop_length=128):
    """compute log-mel spectrogram — shape: (n_mels, time_frames)"""
    # 1. STFT: slide a window, compute FFT at each position
    f, t_stft, Zxx = spectrogram(signal, fs=sr, nperseg=n_fft,
                                  noverlap=n_fft - hop_length, window='hann')
    power = np.abs(Zxx) ** 2   # power spectrogram
    
    # 2. apply mel filterbank
    fb = mel_filterbank(n_mels, n_fft, sr)
    mel = np.dot(fb, power)    # (n_mels, time_frames)
    
    # 3. log compression
    log_mel = np.log(mel + 1e-9)
    return log_mel


# --- compute mel spectrograms ---
normal_mel  = compute_mel_spectrogram(peak_normalized, SAMPLE_RATE)
print(f'Mel spectrogram shape: {normal_mel.shape}')
print(f'  rows = {normal_mel.shape[0]} mel frequency bands')
print(f'  cols = {normal_mel.shape[1]} time frames')
print(f'  each frame = {128/SAMPLE_RATE*1000:.1f} ms of audio')

In [ ]:
# ---- plot: waveform and mel spectrogram side by side ----
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

axes[0].plot(t, peak_normalized, color='steelblue', linewidth=0.5)
axes[0].set_title('Waveform (time domain)')
axes[0].set_xlabel('Time (s)')
axes[0].set_ylabel('Amplitude')

img = axes[1].imshow(normal_mel, aspect='auto', origin='lower',
                     cmap='magma', interpolation='nearest')
axes[1].set_title('Log-Mel Spectrogram (what the model sees)')
axes[1].set_xlabel('Time frames')
axes[1].set_ylabel('Mel frequency bands')
plt.colorbar(img, ax=axes[1], label='log energy')

plt.suptitle('Same audio — two representations', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('the spectrogram is just a 2D matrix of numbers -- perfect for a neural network!')

---
# PART II — Full Pipeline: Normal Speech → Whisper Voice

## Overview

```
Normal Speech (normal.wav)
        |
        v  [feature extraction]
  Normal Mel Spectrogram
        |
        v  [Mel-Mel Model (neural network)]
  Predicted Whisper Mel
        |
        v  [Vocoder]
  Whisper Audio (whisper_audio.wav)
```

The idea is simple:
- We have **pairs** of the same person saying the same thing: once normally, once whispering
- We train a network to learn the mapping: `normal_mel → whisper_mel`
- At inference, we pass new normal speech and get whisper mel out
- Then a vocoder converts mel back to audio

---
## Step 1 — Dataset Preparation

We need **paired** recordings: the same utterance spoken normally AND in a whisper.

```
dataset/
├── normal/
│   ├── 001.wav   <- "hello how are you" (normal voice)
│   ├── 002.wav
│   └── ...
└── whisper/
    ├── 001.wav   <- "hello how are you" (whisper)
    ├── 002.wav
    └── ...
```

Key requirements:
- Same speaker, same text, same duration (aligned)
- Trimmed (no trailing silence) — this is where `trim_speech.py` helps!
- Normalized to the same loudness level

In [ ]:
# ============================================================
# PRACTICAL EXAMPLE: Simulate a paired dataset
# ============================================================

np.random.seed(42)

def simulate_normal_speech(duration=2.0, sr=16000):
    """simulate normal speech: rich harmonics, higher amplitude"""
    t = np.linspace(0, duration, int(sr * duration))
    signal = (
        0.6 * np.sin(2 * np.pi * 150  * t) +   # fundamental frequency
        0.4 * np.sin(2 * np.pi * 300  * t) +   # 2nd harmonic
        0.3 * np.sin(2 * np.pi * 900  * t) +   # 1st formant region
        0.2 * np.sin(2 * np.pi * 2200 * t) +   # 2nd formant region
        0.03 * np.random.randn(len(t))          # small background noise
    )
    # add envelope (speech isn't constant -- it has pauses)
    envelope = np.ones(len(t))
    envelope[:int(0.1*sr)] = np.linspace(0, 1, int(0.1*sr))  # fade in
    envelope[-int(0.1*sr):] = np.linspace(1, 0, int(0.1*sr)) # fade out
    return signal * envelope / (np.max(np.abs(signal)) + 1e-8)

def simulate_whisper(duration=2.0, sr=16000):
    """simulate whisper: no voiced harmonics, mostly noise + frication"""
    t = np.linspace(0, duration, int(sr * duration))
    # whisper = shaped noise (no pitch/periodicity)
    noise = np.random.randn(len(t))
    # shape the noise with formant-like filters (simplified)
    signal = (
        0.4 * noise * np.abs(np.sin(2 * np.pi * 1.5 * t)) +   # modulated noise
        0.2 * np.sin(2 * np.pi * 2200 * t) +                  # high-freq content
        0.15 * np.sin(2 * np.pi * 3500 * t)                   # fricative energy
    )
    envelope = np.ones(len(t))
    envelope[:int(0.1*sr)] = np.linspace(0, 1, int(0.1*sr))
    envelope[-int(0.1*sr):] = np.linspace(1, 0, int(0.1*sr))
    return signal * envelope / (np.max(np.abs(signal)) + 1e-8)

# simulate 5 paired utterances
N_SAMPLES = 5
dataset_normal  = [simulate_normal_speech() for _ in range(N_SAMPLES)]
dataset_whisper = [simulate_whisper()        for _ in range(N_SAMPLES)]

print(f'Dataset created: {N_SAMPLES} paired utterances')
print(f'Each utterance: {len(dataset_normal[0])} samples = {len(dataset_normal[0])/SAMPLE_RATE:.1f} sec')

In [ ]:
# ---- compare normal vs whisper waveform ----
t_plot = np.linspace(0, 2.0, len(dataset_normal[0]))

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)

axes[0].plot(t_plot, dataset_normal[0], color='steelblue', linewidth=0.5)
axes[0].set_title('Normal speech waveform (periodic, rhythmic structure)')
axes[0].set_ylabel('Amplitude')
axes[0].set_ylim(-1.2, 1.2)

axes[1].plot(t_plot, dataset_whisper[0], color='orangered', linewidth=0.5)
axes[1].set_title('Whisper waveform (aperiodic, noisy -- no vocal cord vibration)')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylim(-1.2, 1.2)

plt.suptitle('Paired Utterance #1: Same sentence, different voice types', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 2 — Feature Extraction

Convert every `.wav` file → Mel spectrogram.
Now each audio file becomes a **2D matrix** of shape `(n_mels, time_frames)`.

The model will learn: `normal_mel  →  whisper_mel`

In [ ]:
# ============================================================
# PRACTICAL EXAMPLE: Extract mel features for the whole dataset
# ============================================================

N_MELS = 80

normal_mels  = [compute_mel_spectrogram(s, SAMPLE_RATE, n_mels=N_MELS) for s in dataset_normal]
whisper_mels = [compute_mel_spectrogram(s, SAMPLE_RATE, n_mels=N_MELS) for s in dataset_whisper]

print(f'Feature extraction done!')
print(f'Each mel: shape = {normal_mels[0].shape}  ({N_MELS} mel bands x {normal_mels[0].shape[1]} time frames)')

# --- visualize paired mel spectrograms ---
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

im1 = axes[0].imshow(normal_mels[0], aspect='auto', origin='lower', cmap='magma')
axes[0].set_title('normal_mel  (input to the model)')
axes[0].set_xlabel('Time frames')
axes[0].set_ylabel('Mel band index')
plt.colorbar(im1, ax=axes[0], label='log energy')

im2 = axes[1].imshow(whisper_mels[0], aspect='auto', origin='lower', cmap='magma')
axes[1].set_title('whisper_mel  (target -- what we want the model to produce)')
axes[1].set_xlabel('Time frames')
axes[1].set_ylabel('Mel band index')
plt.colorbar(im2, ax=axes[1], label='log energy')

plt.suptitle('Feature Extraction: normal.wav → normal_mel   |   whisper.wav → whisper_mel',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('notice the difference:')
print('  normal_mel  -> energy concentrated in lower mel bands (voiced harmonics)')
print('  whisper_mel -> energy more spread across all bands (frication/noise source)')

---
## Step 3 — Mel-Mel Model Architecture

The model takes a **normal mel spectrogram** and outputs a **whisper mel spectrogram** of the same shape.

This is an **encoder–decoder** style architecture (like a U-Net):

```
normal_mel [80 x T]
      |
   Encoder   <- compress features, find patterns
      |
  Bottleneck <- compact representation
      |
   Decoder   <- reconstruct whisper mel from compact representation
      |
predicted_whisper_mel [80 x T]
```

We'll build a simple version using **Conv1d layers** (treating the mel as a sequence of 80-dim vectors over time).

In [ ]:
# ============================================================
# Mel-Mel Model in PyTorch
# (if torch not installed, we show the architecture only)
# ============================================================

try:
    import torch
    import torch.nn as nn
    TORCH_OK = True
    print('PyTorch found:', torch.__version__)
except ImportError:
    TORCH_OK = False
    print('PyTorch not installed -- run: pip install torch')
    print('Showing architecture as text only')

In [ ]:
if TORCH_OK:
    class MelToMelModel(nn.Module):
        """
        Simple Mel-to-Mel conversion model.
        Input:  (batch, n_mels, time)  <- normal mel
        Output: (batch, n_mels, time)  <- predicted whisper mel
        """
        def __init__(self, n_mels=80, hidden=128):
            super().__init__()
            
            # --- encoder: extract features, reduce time resolution ---
            self.encoder = nn.Sequential(
                nn.Conv1d(n_mels, hidden, kernel_size=3, padding=1),   # (batch, 128, T)
                nn.ReLU(),
                nn.Conv1d(hidden, hidden, kernel_size=3, padding=1),   # (batch, 128, T)
                nn.ReLU(),
                nn.Conv1d(hidden, hidden*2, kernel_size=3, stride=2, padding=1),  # downsample
                nn.ReLU(),
            )
            
            # --- bottleneck: deeper feature learning ---
            self.bottleneck = nn.Sequential(
                nn.Conv1d(hidden*2, hidden*2, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Conv1d(hidden*2, hidden*2, kernel_size=3, padding=1),
                nn.ReLU(),
            )
            
            # --- decoder: upsample back, produce whisper mel ---
            self.decoder = nn.Sequential(
                nn.ConvTranspose1d(hidden*2, hidden, kernel_size=4, stride=2, padding=1),  # upsample
                nn.ReLU(),
                nn.Conv1d(hidden, hidden, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Conv1d(hidden, n_mels, kernel_size=3, padding=1),   # back to (batch, 80, T)
            )
        
        def forward(self, x):
            x = self.encoder(x)
            x = self.bottleneck(x)
            x = self.decoder(x)
            return x

    model = MelToMelModel(n_mels=N_MELS, hidden=128)
    print('Model architecture:')
    print(model)
    
    total_params = sum(p.numel() for p in model.parameters())
    print(f'\nTotal parameters: {total_params:,}')

else:
    print("""
    MelToMelModel architecture (Conv1d encoder-decoder):

    INPUT: (batch, 80, T)  -- normal mel
    
    ENCODER:
      Conv1d(80  -> 128, k=3) + ReLU
      Conv1d(128 -> 128, k=3) + ReLU
      Conv1d(128 -> 256, k=3, stride=2) + ReLU   <- downsample
    
    BOTTLENECK:
      Conv1d(256 -> 256, k=3) + ReLU
      Conv1d(256 -> 256, k=3) + ReLU
    
    DECODER:
      ConvTranspose1d(256 -> 128, k=4, stride=2) + ReLU  <- upsample
      Conv1d(128 -> 128, k=3) + ReLU
      Conv1d(128 -> 80,  k=3)                            <- back to mel shape
    
    OUTPUT: (batch, 80, T)  -- predicted whisper mel
    """)

---
## Step 4 — Training

**Goal:** Minimize the difference between `predicted_whisper_mel` and the **real** `whisper_mel`.

**Loss function:** MSE (Mean Squared Error)
$$\mathcal{L} = \frac{1}{N} \sum (\hat{y} - y)^2$$

**Training loop:**
```
for each epoch:
    for each (normal_mel, whisper_mel) pair in dataset:
        predicted = model(normal_mel)
        loss = MSE(predicted, whisper_mel)
        loss.backward()         <- compute gradients
        optimizer.step()        <- update weights
```

In [ ]:
# ============================================================
# PRACTICAL EXAMPLE: Training loop (toy example)
# ============================================================

if TORCH_OK:
    import torch.optim as optim
    
    # convert numpy mels to torch tensors
    # shape: (batch=5, n_mels=80, time=T)
    T = min(m.shape[1] for m in normal_mels)  # use shortest to keep same length
    X = torch.FloatTensor(np.stack([m[:, :T] for m in normal_mels]))   # input
    Y = torch.FloatTensor(np.stack([m[:, :T] for m in whisper_mels]))  # target
    
    print(f'Training data shapes:')
    print(f'  X (normal_mel)  : {X.shape}')
    print(f'  Y (whisper_mel) : {Y.shape}')
    
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()
    
    N_EPOCHS = 100
    loss_history = []
    
    model.train()
    for epoch in range(N_EPOCHS):
        optimizer.zero_grad()
        
        predicted = model(X)           # forward pass
        
        # predicted might have slightly different T due to stride/upsample
        # trim to same length
        t_min = min(predicted.shape[2], Y.shape[2])
        loss = criterion(predicted[:, :, :t_min], Y[:, :, :t_min])
        
        loss.backward()                # backward pass
        optimizer.step()               # update weights
        
        loss_history.append(loss.item())
        
        if (epoch + 1) % 20 == 0:
            print(f'  Epoch {epoch+1:3d}/{N_EPOCHS} | Loss: {loss.item():.4f}')
    
    print('\nTraining done!')

else:
    # simulate a loss curve for illustration
    N_EPOCHS = 100
    loss_history = 2.0 * np.exp(-np.linspace(0, 3, N_EPOCHS)) + 0.3 + 0.05*np.random.randn(N_EPOCHS)
    loss_history = np.maximum(loss_history, 0.25)
    print('Showing simulated loss curve (PyTorch not installed)')

In [ ]:
# ---- plot training loss curve ----
plt.figure(figsize=(10, 4))
plt.plot(loss_history, color='steelblue', linewidth=1.5, label='Training Loss (MSE)')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Progress — Loss should go down over time')
plt.legend()
plt.tight_layout()
plt.show()

print('Good sign: loss is decreasing -- the model is learning!')
print('If loss stops improving early, try: lower learning rate, more data, bigger model')

---
## Step 5 — Inference

After training, we can take **any new normal speech** and convert it to whisper mel:

```python
new_normal_mel = compute_mel_spectrogram(new_normal_wav)
predicted_whisper_mel = model(new_normal_mel)   # no gradients needed
```

In [ ]:
# ============================================================
# PRACTICAL EXAMPLE: Inference on a new utterance
# ============================================================

# simulate a new normal speech recording (not seen during training)
np.random.seed(99)
new_normal_wav = simulate_normal_speech(duration=2.0)
new_normal_mel = compute_mel_spectrogram(new_normal_wav, SAMPLE_RATE, n_mels=N_MELS)

if TORCH_OK:
    model.eval()   # turn off dropout/batchnorm training mode
    with torch.no_grad():   # no gradient computation needed at inference
        x_infer = torch.FloatTensor(new_normal_mel).unsqueeze(0)  # add batch dim
        predicted_whisper_mel_tensor = model(x_infer)
        predicted_whisper_mel = predicted_whisper_mel_tensor.squeeze(0).numpy()  # back to numpy
    print(f'Inference done!')
    print(f'  Input  shape: {x_infer.shape}')
    print(f'  Output shape: {predicted_whisper_mel_tensor.shape}')
else:
    # fake prediction: normal mel + noise shift (to simulate what a trained model might do)
    predicted_whisper_mel = new_normal_mel * 0.7 + 0.3 * np.random.randn(*new_normal_mel.shape)
    print('Showing simulated inference (PyTorch not installed)')

# ---- compare: input mel vs predicted whisper mel ----
fig, axes = plt.subplots(1, 2, figsize=(15, 4))

im1 = axes[0].imshow(new_normal_mel, aspect='auto', origin='lower', cmap='magma')
axes[0].set_title('Input: new_normal_mel')
axes[0].set_xlabel('Time frames'); axes[0].set_ylabel('Mel band')
plt.colorbar(im1, ax=axes[0])

im2 = axes[1].imshow(predicted_whisper_mel, aspect='auto', origin='lower', cmap='magma')
axes[1].set_title('Output: predicted_whisper_mel')
axes[1].set_xlabel('Time frames'); axes[1].set_ylabel('Mel band')
plt.colorbar(im2, ax=axes[1])

plt.suptitle('Inference: New normal speech → Predicted whisper mel', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 6 — Vocoder: Mel Spectrogram → Audio

The model outputs a mel spectrogram. But we want **audio** (a `.wav` file).
A **vocoder** does this inverse step:

```
whisper_mel [80 x T]  →  [Vocoder]  →  whisper_audio.wav
```

**Common vocoders:**

| Vocoder | Type | Quality | Speed |
|---|---|---|---|
| Griffin-Lim | Classical (iterative phase estimation) | OK | Fast |
| WaveNet | Neural (autoregressive) | Excellent | Slow |
| HiFi-GAN | Neural (GAN) | Excellent | Fast |
| WaveGlow | Neural (flow-based) | Very good | Medium |

We'll demo **Griffin-Lim** as it requires no additional models.

In [ ]:
# ============================================================
# PRACTICAL EXAMPLE: Griffin-Lim vocoder (mel → waveform)
# ============================================================

def griffin_lim(mel_spec, sr=16000, n_fft=512, hop_length=128, n_mels=80, n_iter=60):
    """
    Convert log-mel spectrogram back to waveform using Griffin-Lim algorithm.
    This is a simplified version -- real vocoders (HiFi-GAN) are much better.
    """
    # 1. invert the log compression
    mel_power = np.exp(mel_spec)  # (n_mels, T)
    
    # 2. invert mel filterbank (pseudo-inverse)
    fb = mel_filterbank(n_mels, n_fft, sr)
    fb_pinv = np.linalg.pinv(fb)                  # (n_fft//2+1, n_mels)
    power = np.maximum(fb_pinv @ mel_power, 0)    # (n_fft//2+1, T)  linear spectrogram
    magnitude = np.sqrt(power + 1e-9)
    
    # 3. Griffin-Lim: iteratively estimate phase
    # start with random phase
    phase = np.exp(2j * np.pi * np.random.rand(*magnitude.shape))
    
    for _ in range(n_iter):
        # combine magnitude with current phase estimate → complex spectrum
        complex_spec = magnitude * phase
        
        # iSTFT → time domain
        # manual overlap-add
        n_frames = complex_spec.shape[1]
        signal = np.zeros((n_frames - 1) * hop_length + n_fft)
        window = np.hanning(n_fft)
        
        for i in range(n_frames):
            frame = np.real(np.fft.irfft(complex_spec[:, i], n=n_fft)) * window
            start = i * hop_length
            signal[start:start + n_fft] += frame
        
        # STFT of reconstructed signal → update phase
        n_frames2 = (len(signal) - n_fft) // hop_length + 1
        new_phase = np.zeros_like(phase)
        for i in range(min(n_frames, n_frames2)):
            frame = signal[i*hop_length : i*hop_length + n_fft] * window
            spec  = np.fft.rfft(frame, n=n_fft)
            new_phase[:, i] = np.exp(1j * np.angle(spec))
        phase = new_phase
    
    # final reconstruction
    n_frames = complex_spec.shape[1]
    out = np.zeros((n_frames - 1) * hop_length + n_fft)
    for i in range(n_frames):
        frame = np.real(np.fft.irfft(magnitude[:, i] * phase[:, i], n=n_fft)) * window
        out[i*hop_length : i*hop_length + n_fft] += frame
    
    # normalize output
    out = out / (np.max(np.abs(out)) + 1e-8)
    return out.astype(np.float32)

print('Running Griffin-Lim vocoder (this takes a few seconds)...')
reconstructed_whisper = griffin_lim(predicted_whisper_mel, sr=SAMPLE_RATE, n_mels=N_MELS, n_iter=30)
print(f'Done! Output audio: {len(reconstructed_whisper)} samples = {len(reconstructed_whisper)/SAMPLE_RATE:.2f} sec')

In [ ]:
# ---- final comparison: input normal speech vs reconstructed whisper ----
t_in  = np.linspace(0, len(new_normal_wav)/SAMPLE_RATE, len(new_normal_wav))
t_out = np.linspace(0, len(reconstructed_whisper)/SAMPLE_RATE, len(reconstructed_whisper))

fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=False)

axes[0].plot(t_in, new_normal_wav, color='steelblue', linewidth=0.5)
axes[0].set_title('Input: Normal speech (normal.wav)')
axes[0].set_ylabel('Amplitude')
axes[0].set_ylim(-1.2, 1.2)

axes[1].plot(t_out, reconstructed_whisper, color='orangered', linewidth=0.5)
axes[1].set_title('Output: Reconstructed whisper (whisper_audio.wav via Griffin-Lim vocoder)')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylim(-1.2, 1.2)

plt.suptitle('Full Pipeline: Normal Speech → Whisper Audio', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# save output WAV
out_int16 = (reconstructed_whisper * 32767).astype(np.int16)
wavfile.write('whisper_output.wav', SAMPLE_RATE, out_int16)
print('saved: whisper_output.wav')

---
## Full Pipeline Summary

```
Normal Speech (normal.wav)
        |
        |  [1] normalize amplitude
        |
        |  [2] compute Log-Mel Spectrogram
        v
  normal_mel  (shape: 80 x T)
        |
        |  [3] Mel-Mel Model (trained Conv1d encoder-decoder)
        v
  predicted_whisper_mel  (shape: 80 x T)
        |
        |  [6] Vocoder (Griffin-Lim or HiFi-GAN)
        v
  whisper_audio.wav
```

**Training requires:**
- Paired recordings: `normal.wav` ↔ `whisper.wav` (same speaker, same text)
- Feature extraction for both
- MSE loss between `predicted_whisper_mel` and real `whisper_mel`

**Key takeaways:**
- Audio is just numbers (float32 samples)
- Mel spectrogram = 2D image of audio (easy for neural nets)
- The Mel-Mel model learns the **spectral mapping** from normal to whisper
- The vocoder converts the predicted mel back to audio

In [ ]:
# ============================================================
# Full pipeline: put it all together in one function
# ============================================================

def run_full_pipeline(input_wav_array, model=None, sr=16000, n_mels=80):
    """
    Full inference pipeline:
    input_wav_array  ->  whisper_wav_array
    """
    print('[1] Normalizing audio...')
    normalized = input_wav_array / (np.max(np.abs(input_wav_array)) + 1e-8)
    
    print('[2] Extracting Mel spectrogram...')
    mel = compute_mel_spectrogram(normalized, sr, n_mels=n_mels)
    print(f'    mel shape: {mel.shape}')
    
    print('[3] Running Mel-Mel model...')
    if model is not None and TORCH_OK:
        model.eval()
        with torch.no_grad():
            x = torch.FloatTensor(mel).unsqueeze(0)
            pred_mel = model(x).squeeze(0).numpy()
    else:
        # dummy: use input mel as predicted (untrained)
        pred_mel = mel
    print(f'    predicted whisper mel shape: {pred_mel.shape}')
    
    print('[4] Running Vocoder (Griffin-Lim)...')
    whisper_audio = griffin_lim(pred_mel, sr=sr, n_mels=n_mels, n_iter=30)
    print(f'    output audio: {len(whisper_audio)} samples = {len(whisper_audio)/sr:.2f} sec')
    
    return whisper_audio, mel, pred_mel


# run the full pipeline!
np.random.seed(7)
test_normal_wav = simulate_normal_speech(duration=2.0)

output_audio, in_mel, out_mel = run_full_pipeline(
    test_normal_wav,
    model=model if TORCH_OK else None
)

print('\nPipeline complete!')

In [ ]:
# ---- final visualization: all stages in one figure ----
fig = plt.figure(figsize=(16, 10))
gs = gridspec.GridSpec(3, 2, figure=fig, hspace=0.5, wspace=0.35)

t_in = np.linspace(0, len(test_normal_wav)/SAMPLE_RATE, len(test_normal_wav))
t_out = np.linspace(0, len(output_audio)/SAMPLE_RATE, len(output_audio))

# 1. input waveform
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(t_in, test_normal_wav, color='steelblue', linewidth=0.5)
ax1.set_title('[1] Input: Normal Speech Waveform')
ax1.set_ylabel('Amplitude'); ax1.set_xlabel('Time (s)')

# 2. input mel
ax2 = fig.add_subplot(gs[0, 1])
im2 = ax2.imshow(in_mel, aspect='auto', origin='lower', cmap='magma')
ax2.set_title('[2] Normal Mel Spectrogram')
ax2.set_ylabel('Mel band'); ax2.set_xlabel('Time frame')
plt.colorbar(im2, ax=ax2, shrink=0.8)

# 3. predicted whisper mel
ax3 = fig.add_subplot(gs[1, 0])
im3 = ax3.imshow(out_mel, aspect='auto', origin='lower', cmap='magma')
ax3.set_title('[3] Predicted Whisper Mel  (model output)')
ax3.set_ylabel('Mel band'); ax3.set_xlabel('Time frame')
plt.colorbar(im3, ax=ax3, shrink=0.8)

# 4. output waveform
ax4 = fig.add_subplot(gs[1, 1])
ax4.plot(t_out, output_audio, color='orangered', linewidth=0.5)
ax4.set_title('[4] Output: Whisper Audio Waveform  (after Vocoder)')
ax4.set_ylabel('Amplitude'); ax4.set_xlabel('Time (s)')

# 5. flow diagram
ax5 = fig.add_subplot(gs[2, :])
ax5.axis('off')
flow_text = (
    'FULL PIPELINE FLOW\n\n'
    'Normal Speech (normal.wav)'
    '  →  [normalize]'
    '  →  [Mel extraction]'
    '  →  normal_mel'
    '  →  [Mel-Mel Model]'
    '  →  predicted_whisper_mel'
    '  →  [Vocoder]'
    '  →  whisper_audio.wav'
)
ax5.text(0.5, 0.5, flow_text, transform=ax5.transAxes,
         ha='center', va='center', fontsize=11,
         bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', edgecolor='gray'))

plt.suptitle('Complete Audio Pipeline: Normal Speech → Whisper Voice',
             fontsize=14, fontweight='bold')
plt.savefig('pipeline_overview.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved: pipeline_overview.png')